[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20II%20-%20The%20End-to-End%20Supply%20Chain%20%28Problems%2047-101%29/C.%20Transportation%2C%20Routing%20%26%20Freight%20Management%20%28The%20Physical%20Internet%29/076.%20The%20VRP%20with%20Split%20Deliveries%20%28SDVRP%29/P76-Tier-6_executed.ipynb) [![Open in SageMaker](https://img.shields.io/badge/Open%20in-SageMaker-orange?logo=amazon-aws)](https://aws.amazon.com/sagemaker/) [![Open in Azure](https://img.shields.io/badge/Open%20in-Azure%20Notebooks-blue?logo=microsoft-azure)](https://notebooks.azure.com/)

---

# 76. The Vehicle Routing Problem with Split Deliveries (SDVRP)
## Tier 6: An Auction-Based Multi-Agent Simulation

This notebook represents customer demand as smaller delivery tasks and allocates those tasks to vehicle agents through a central auctioneer. Each agent submits a cost-based bid; the auctioneer awards the task to the lowest bidder that can carry it on a vehicle trip. The simulation then explores how task allocation changes when agent costs change and when agents may transfer tasks.

### Learning objectives

By the end of the notebook, you should be able to:

- Explain how split deliveries can be represented as smaller tasks.
- Trace how a vehicle agent estimates a bid and how an auctioneer selects a winner.
- Relate vehicle capacity to repeated depot-to-customer trips.
- Interpret route, assignment, utilization, and workload statistics without treating them as proof of global optimality.
- Distinguish the implemented auction simulation from a complete Contract Net Protocol or a real-time autonomous logistics system.

### Scope and modeling assumptions

- The depot and customer locations are points in a two-dimensional Euclidean plane. Distances are measured in coordinate units; no road network or travel-time model is used.
- Customer demand is divided into tasks of at most 20 units. A task is assigned whole to one vehicle, while different tasks for the same customer may be assigned to different vehicles. Thus the instance permits split deliveries at task boundaries, not arbitrary fractional task assignment.
- A vehicle may make repeated trips. Its capacity applies to each trip; when another task would exceed the remaining trip capacity, its planned route returns to the depot before serving that task.
- The auctioneer is centralized, bids are evaluated sequentially, and the lowest eligible bid wins. Bid adjustments and random noise are illustrative rules, not a guarantee of minimum total route distance.
- Re-bidding is a simple task-transfer simulation after cost changes. It does not model traffic on a road network, communication failures, time windows, service times, or a full message-based Contract Net Protocol.
- The example uses a fixed random seed for reproducibility. Its small scale makes it suitable for tracing the mechanism, not for establishing performance on large routing instances.

### Solution sequence

1. Define the SDVRP instance and Euclidean distance matrix.
2. Define delivery tasks and vehicle agents, including their bidding and route-planning rules.
3. Let a central auctioneer allocate tasks in sequence.
4. Change agent cost parameters and examine possible task transfers.
5. Visualize and summarize the resulting assignments and routes.

In [ ]:
import random

import matplotlib.pyplot as plt
import numpy as np

# Fix both random number generators so that illustrative bid noise is reproducible.
np.random.seed(76)
random.seed(76)

print("Libraries imported successfully for the multi-agent simulation.")

In [ ]:
class SDVRPInstance:
    """Store an SDVRP instance and calculate its Euclidean distances."""

    def __init__(self, depot_coords, customer_coords, demands, vehicle_capacities):
        if len(customer_coords) != len(demands):
            raise ValueError("Each customer must have exactly one demand value.")
        if any(demand < 0 for demand in demands):
            raise ValueError("Customer demands must be non-negative.")
        if any(capacity <= 0 for capacity in vehicle_capacities):
            raise ValueError("Vehicle capacities must be positive.")

        self.depot = depot_coords
        self.customers = customer_coords
        self.demands = demands
        self.capacities = vehicle_capacities
        self.n_customers = len(customer_coords)
        self.n_vehicles = len(vehicle_capacities)

    def calculate_distance_matrix(self):
        """Return distances between the depot and all customer locations."""
        all_nodes = [self.depot] + self.customers
        n_nodes = len(all_nodes)
        distances = np.zeros((n_nodes, n_nodes), dtype=float)

        for i in range(n_nodes):
            for j in range(n_nodes):
                x1, y1 = all_nodes[i]
                x2, y2 = all_nodes[j]
                distances[i, j] = np.sqrt((x2 - x1) ** 2 + (y2 - y1) ** 2)

        return distances


# This instance uses the same four customers and two vehicles throughout the notebook.
depot_coords = (0, 0)
customer_coords = [(10, 15), (20, 5), (15, 20), (25, 10)]
demands = [70, 130, 60, 80]
vehicle_capacities = [100, 100]

instance = SDVRPInstance(
    depot_coords,
    customer_coords,
    demands,
    vehicle_capacities,
)
distance_matrix = instance.calculate_distance_matrix()

print("SDVRP instance for the multi-agent simulation:")
print(f"- Depot: {depot_coords}")
print(f"- Customers: {instance.n_customers} with demands: {demands}")
print(f"- Vehicles: {instance.n_vehicles} with capacities: {vehicle_capacities}")
print(f"- Total demand: {sum(demands)} units")
print(f"- Total single-trip capacity: {sum(vehicle_capacities)} units")

## Modeling the task auction

### Intuition

A customer's demand may be larger than the capacity available on one vehicle trip. The task generator therefore breaks demand into smaller pieces. For each piece, every vehicle that can carry it submits a bid, and the auctioneer assigns it to the vehicle with the lowest bid. If that vehicle does not have enough unused capacity on its current trip, it returns to the depot before serving the task.

### Sets, data, and decisions

Let $C$ be the set of customers, $V$ the set of vehicles, and $K$ the set of delivery tasks created from customer demand. Task $k \in K$ has customer $c(k)$, quantity $q_k$, and location at that customer's node. Vehicle $v \in V$ has trip capacity $Q_v$. The depot is node $0$; customer $i$ is node $i+1$ in the distance matrix. Let $d_{ij}$ be the Euclidean distance from node $i$ to node $j$.

The allocation decision for a task is its winning vehicle, denoted $a(k)$. Before bidding on task $k$, vehicle $v$ has current location $\ell_v$ and used capacity $L_v$ on its current trip. Its route increment is

$$
\Delta_{vk} =
\begin{cases}
d_{\ell_v,c(k)+1}, & L_v + q_k \leq Q_v, \\
d_{\ell_v,0} + d_{0,c(k)+1}, & L_v + q_k > Q_v.
\end{cases}
$$

The second case includes the return to the depot and the next depot-to-customer leg. The agent converts this increment into a bid using its cost-per-distance parameter and illustrative adjustment factors. The auctioneer chooses

$$
a(k) \in \arg\min_{v \in V_k} b_{vk},
\qquad
V_k = \{v \in V : q_k \leq Q_v \text{ and the agent passes the fuel eligibility check}\},
$$

where $b_{vk}$ is vehicle $v$'s bid and $V_k$ is the set of eligible bidders. Ties are resolved by the order in which agents are registered because Python's `min` returns the first minimum.

### What this model does and does not optimize

The sequence of auctions is a heuristic allocation rule: each task is decided using the agents' current states, rather than by jointly optimizing every task and route. Bid adjustments for utilization, workload, risk tolerance, and competitiveness are agent-level scoring rules. Consequently, the winning bids do not establish that the final total route distance is globally minimal.

The code below implements the current location, used trip capacity, route increment, bid, and eligibility concepts directly. The task-level auction is the notebook's allocation mechanism; no separate exact optimization model is solved.

In [ ]:
class DeliveryTask:
    """Represent one indivisible quantity of a customer's demand."""

    def __init__(self, task_id, customer_id, quantity, location, priority="normal"):
        self.task_id = task_id
        self.customer_id = customer_id
        self.quantity = quantity
        self.location = location
        self.priority = priority
        self.assigned_agent = None
        self.bid_history = []

    def add_bid(self, agent_id, bid_amount, confidence):
        """Keep the bid and confidence together for inspecting an auction."""
        self.bid_history.append(
            {
                "agent_id": agent_id,
                "bid_amount": bid_amount,
                "confidence": confidence,
            }
        )

    def get_lowest_bid(self):
        """Return the lowest recorded bid, or None if no agent bid."""
        if not self.bid_history:
            return None
        return min(self.bid_history, key=lambda bid: bid["bid_amount"])

    def __str__(self):
        return (
            f"Task-{self.task_id}: Customer {self.customer_id}, "
            f"quantity {self.quantity}, priority {self.priority}"
        )


class VehicleAgent:
    """Vehicle agent that estimates task costs and maintains a planned route."""

    def __init__(self, agent_id, capacity, initial_location, distance_matrix):
        self.agent_id = agent_id
        self.capacity = capacity
        self.current_location = initial_location
        self.distance_matrix = distance_matrix

        self.current_load = 0
        self.assigned_tasks = []
        self.completed_tasks = []
        self.route = [initial_location]

        self.cost_per_km = 2.5
        self.max_range = 500
        self.current_fuel = 100.0

        self.risk_tolerance = 0.5
        self.competitiveness = 0.7
        self.workload_preference = 0.6

        self.bid_success_history = []
        self.profit_history = []

    def calculate_incremental_distance(self, task):
        """Estimate the added route distance, including a required depot return."""
        customer_node = task.customer_id
        if self.current_load + task.quantity > self.capacity:
            return (
                self.distance_matrix[self.current_location, 0]
                + self.distance_matrix[0, customer_node]
            )
        return self.distance_matrix[self.current_location, customer_node]

    def calculate_bid_cost(self, task):
        """Convert an estimated route increment into an adjusted task bid."""
        base_cost = self.calculate_incremental_distance(task) * self.cost_per_km

        projected_load = self.current_load + task.quantity
        if projected_load > self.capacity:
            projected_load = task.quantity
        utilization = projected_load / self.capacity
        utilization_factor = 1.0 - (utilization - 0.5) ** 2

        workload_factor = 1.0
        if len(self.assigned_tasks) > 3:
            workload_factor = 1.2
        elif len(self.assigned_tasks) == 0:
            workload_factor = 0.8

        risk_adjustment = 1.0 + self.risk_tolerance * 0.1
        competitive_adjustment = 1.0 - self.competitiveness * 0.15

        return (
            base_cost
            * utilization_factor
            * workload_factor
            * risk_adjustment
            * competitive_adjustment
        )

    def calculate_bid_confidence(self, task):
        """Estimate task feasibility from capacity, fuel, experience, and distance."""
        base_confidence = 0.8
        capacity_confidence = 1.0 if task.quantity <= self.capacity else 0.3

        distance_to_task = self.calculate_incremental_distance(task)
        fuel_needed = distance_to_task * 0.1
        fuel_confidence = 1.0 if self.current_fuel >= fuel_needed else 0.2

        experience_factor = min(1.0, 0.7 + len(self.completed_tasks) * 0.05)
        max_distance = float(np.max(self.distance_matrix))
        if max_distance > 0:
            distance_factor = 1.0 - (distance_to_task / max_distance) * 0.2
        else:
            distance_factor = 1.0

        confidence = (
            base_confidence
            * capacity_confidence
            * fuel_confidence
            * experience_factor
            * distance_factor
        )
        return max(0.1, min(1.0, confidence))

    def place_bid(self, task):
        """Submit a cost bid with small reproducible illustrative noise."""
        bid_cost = self.calculate_bid_cost(task)
        confidence = self.calculate_bid_confidence(task)
        noise = np.random.normal(0, 0.05)
        bid_cost *= 1 + noise
        task.add_bid(self.agent_id, bid_cost, confidence)
        return bid_cost, confidence

    def assign_task(self, task):
        """Add a task, returning to the depot first if trip capacity requires it."""
        if task.quantity > self.capacity:
            raise ValueError("A task cannot be assigned to a vehicle smaller than the task.")

        if self.current_load + task.quantity > self.capacity:
            if self.current_location != 0:
                self.route.append(0)
                self.current_location = 0
            self.current_load = 0
            self.current_fuel = 100.0

        customer_node = task.customer_id
        previous_location = self.current_location
        if previous_location != customer_node:
            self.route.append(customer_node)

        travel_distance = self.distance_matrix[previous_location, customer_node]
        self.current_fuel -= travel_distance * 0.1
        self.current_location = customer_node
        self.current_load += task.quantity
        self.assigned_tasks.append(task)
        task.assigned_agent = self.agent_id

    def complete_task(self, task):
        """Record completion without changing the already planned route history."""
        if task in self.assigned_tasks:
            self.assigned_tasks.remove(task)
            self.completed_tasks.append(task)

    def return_to_depot(self):
        """Append a depot return when a caller needs an explicitly closed route."""
        if self.current_location != 0:
            distance = self.distance_matrix[self.current_location, 0]
            self.route.append(0)
            self.current_location = 0
            self.current_fuel -= distance * 0.1

    def calculate_route_distance(self):
        """Measure the planned route, including its final return to the depot."""
        total_distance = sum(
            self.distance_matrix[start, end]
            for start, end in zip(self.route, self.route[1:])
        )
        if self.current_location != 0:
            total_distance += self.distance_matrix[self.current_location, 0]
        return float(total_distance)

    def rebuild_plan(self, tasks):
        """Rebuild this agent's route after its allocated task set changes."""
        self.current_location = 0
        self.current_load = 0
        self.current_fuel = 100.0
        self.route = [0]
        self.assigned_tasks = []
        for task in tasks:
            self.assign_task(task)

    def get_status(self):
        """Return a snapshot used by the reporting and visualization cells."""
        route_for_display = self.route.copy()
        if self.current_location != 0:
            route_for_display.append(0)

        return {
            "agent_id": self.agent_id,
            "current_location": self.current_location,
            "current_load": self.current_load,
            "capacity": self.capacity,
            "utilization": (self.current_load / self.capacity) * 100,
            "assigned_tasks": len(self.assigned_tasks),
            "completed_tasks": len(self.completed_tasks),
            "current_fuel": self.current_fuel,
            "route": route_for_display,
            "route_distance": self.calculate_route_distance(),
        }


print("Delivery-task and vehicle-agent classes are ready.")

### How the auction advances

The auction is sequential rather than a simultaneous solution of all task decisions. After a task is awarded, the winning agent's location and used capacity change. Those updated values affect its bids on later tasks. If a task no longer fits on the current trip, the route increment includes travel back to the depot and then onward to the task; the route planner also records that depot visit.

For a task $k$, the allocation rule is the one-step decision

$$
a(k) = \operatorname*{arg\,min}_{v \in V_k} b_{vk}.
$$

Here, $V_k$ is the set of agents permitted to bid, and $b_{vk}$ is the adjusted bid returned by `VehicleAgent.place_bid`. The update after awarding a task is implemented by `VehicleAgent.assign_task`. Repeating this decision over all tasks is a greedy, state-dependent allocation process; it does not use a Bellman recursion or prove global optimality.

If there are $m$ tasks and $n$ vehicles, collecting and comparing one bid per eligible vehicle for each task requires at most $O(mn)$ bid evaluations. Route-state updates are performed as tasks are assigned. This complexity describes the auction loop, not the quality of its solution.

In [ ]:
class CentralAuctioneer:
    """Coordinate sequential task auctions and collect system metrics."""

    def __init__(self, instance, distance_matrix):
        self.instance = instance
        self.distance_matrix = distance_matrix
        self.agents = []
        self.tasks = []
        self.completed_tasks = []

        self.min_bidding_rounds = 2
        self.max_bidding_rounds = 5
        self.bid_timeout = 5.0

        self.auction_history = []
        self.efficiency_metrics = []

    def register_agent(self, agent):
        """Make an agent available to bid in subsequent auctions."""
        self.agents.append(agent)
        print(f"Agent {agent.agent_id} registered (capacity: {agent.capacity}).")

    def create_tasks(self, demands, granularity=10):
        """Split each demand into tasks no larger than the chosen granularity."""
        if granularity <= 0:
            raise ValueError("Task granularity must be positive.")
        if len(demands) != self.instance.n_customers:
            raise ValueError("Demand values must match the number of customers.")

        self.tasks = []
        task_id = 0
        for customer_index, demand in enumerate(demands):
            if demand < 0:
                raise ValueError("Customer demands must be non-negative.")

            remaining_quantity = demand
            while remaining_quantity > 0:
                task_quantity = min(granularity, remaining_quantity)
                if demand > 100:
                    priority = "high"
                elif demand > 50:
                    priority = "normal"
                else:
                    priority = "low"

                task = DeliveryTask(
                    task_id=task_id,
                    customer_id=customer_index + 1,
                    quantity=task_quantity,
                    location=self.instance.customers[customer_index],
                    priority=priority,
                )
                self.tasks.append(task)
                remaining_quantity -= task_quantity
                task_id += 1

        print(
            f"Created {len(self.tasks)} tasks from "
            f"{len(demands)} customer demands."
        )
        return self.tasks

    def run_auction_round(self, task):
        """Collect eligible bids and award one task to the lowest bidder."""
        print(f"\n--- Auction for {task} ---")
        task.bid_history = []
        bids = []

        for agent in self.agents:
            eligible = (
                task.quantity <= agent.capacity
                and agent.current_fuel > 10
            )
            if eligible:
                bid_cost, confidence = agent.place_bid(task)
                bids.append(
                    {
                        "agent_id": agent.agent_id,
                        "bid_cost": bid_cost,
                        "confidence": confidence,
                    }
                )
                print(
                    f"  Agent {agent.agent_id}: bid = {bid_cost:.2f}, "
                    f"confidence = {confidence:.2f}"
                )
            else:
                print(f"  Agent {agent.agent_id}: not eligible to bid.")

        if not bids:
            print("  No eligible bids; the task remains unassigned.")
            return None

        winning_bid = min(bids, key=lambda bid: bid["bid_cost"])
        winning_agent = next(
            agent
            for agent in self.agents
            if agent.agent_id == winning_bid["agent_id"]
        )
        winning_agent.assign_task(task)

        print(
            f"  Winner: Agent {winning_agent.agent_id} "
            f"with bid {winning_bid['bid_cost']:.2f}."
        )
        self.auction_history.append(
            {
                "task_id": task.task_id,
                "winning_agent": winning_agent.agent_id,
                "winning_bid": winning_bid["bid_cost"],
                "num_bids": len(bids),
            }
        )
        return winning_agent

    def run_complete_auction(self):
        """Run one sequential auction for each task, up to the round limit."""
        print("\n=== COMPLETE AUCTION ===")
        print(f"Agents: {len(self.agents)}, tasks: {len(self.tasks)}")

        unassigned_tasks = self.tasks.copy()
        assigned_tasks = []
        auction_round = 0

        while unassigned_tasks and auction_round < self.max_bidding_rounds:
            auction_round += 1
            print(f"\n--- Auction pass {auction_round} ---")
            remaining_tasks = []

            for task in unassigned_tasks:
                winning_agent = self.run_auction_round(task)
                if winning_agent is None:
                    remaining_tasks.append(task)
                else:
                    assigned_tasks.append(task)

            unassigned_tasks = remaining_tasks

        print("\n=== AUCTION RESULTS ===")
        print(f"Assigned tasks: {len(assigned_tasks)}")
        print(f"Unassigned tasks: {len(unassigned_tasks)}")
        print(f"Auction passes used: {auction_round}")
        return assigned_tasks, unassigned_tasks

    def calculate_system_metrics(self):
        """Summarize distance, current-trip utilization, and task allocation."""
        total_distance = sum(
            agent.calculate_route_distance() for agent in self.agents
        )
        total_utilization = sum(
            (agent.current_load / agent.capacity) * 100
            for agent in self.agents
        )
        total_tasks_assigned = sum(
            len(agent.assigned_tasks) for agent in self.agents
        )
        total_tasks_completed = sum(
            len(agent.completed_tasks) for agent in self.agents
        )

        avg_utilization = (
            total_utilization / len(self.agents) if self.agents else 0.0
        )
        metrics = {
            "total_distance": total_distance,
            "avg_utilization": avg_utilization,
            "total_tasks_assigned": total_tasks_assigned,
            "total_tasks_completed": total_tasks_completed,
            "num_agents": len(self.agents),
            "num_tasks": len(self.tasks),
            "assignment_rate": (
                total_tasks_assigned / len(self.tasks) * 100
                if self.tasks
                else 0.0
            ),
        }
        self.efficiency_metrics.append(metrics)
        return metrics


print("Central auctioneer class is ready.")

## Run the multi-agent simulation

The simulation creates one agent for each vehicle, generates 20-unit tasks, and submits each task to the auctioneer in sequence. A task is eligible for any vehicle whose capacity is at least as large as the task. The used capacity is tracked separately for each trip, so a vehicle can serve more total demand than its single-trip capacity by returning to the depot between trips.

The total distance reported below closes each planned vehicle route with a return to the depot. Current-trip utilization is the used capacity on the agent's final planned trip divided by its capacity; it is not the utilization of every trip combined.

In [ ]:
def run_multi_agent_simulation(instance, distance_matrix):
    """Create agents, auction the tasks, and return the resulting state."""
    print("=== MULTI-AGENT SYSTEM SIMULATION ===")
    auctioneer = CentralAuctioneer(instance, distance_matrix)

    for agent_id, capacity in enumerate(instance.capacities):
        agent = VehicleAgent(
            agent_id=agent_id,
            capacity=capacity,
            initial_location=0,
            distance_matrix=distance_matrix,
        )
        auctioneer.register_agent(agent)

    auctioneer.create_tasks(instance.demands, granularity=20)
    assigned_tasks, unassigned_tasks = auctioneer.run_complete_auction()
    final_metrics = auctioneer.calculate_system_metrics()
    agent_statuses = [agent.get_status() for agent in auctioneer.agents]

    print("\n=== FINAL SYSTEM STATE ===")
    print(f"Total route distance: {final_metrics['total_distance']:.2f} distance units")
    print(f"Mean current-trip utilization: {final_metrics['avg_utilization']:.1f}%")
    print(f"Task assignment rate: {final_metrics['assignment_rate']:.1f}%")
    print(f"Tasks assigned: {final_metrics['total_tasks_assigned']}")
    print(f"Tasks unassigned: {len(unassigned_tasks)}")

    assigned_quantity = sum(task.quantity for task in assigned_tasks)
    assert assigned_quantity <= sum(instance.demands)
    assert all(
        task.quantity <= agent.capacity
        for agent in auctioneer.agents
        for task in agent.assigned_tasks
    )
    assert sum(len(agent.assigned_tasks) for agent in auctioneer.agents) == len(
        assigned_tasks
    )

    return auctioneer, agent_statuses, final_metrics


auctioneer, agent_statuses, final_metrics = run_multi_agent_simulation(
    instance,
    distance_matrix,
)

## Dynamic cost changes and task transfers

This experiment changes agent cost parameters over three periods. It then checks whether an agent with a relatively large task count can transfer one task to a less-loaded agent when the receiving agent's estimated bid cost is at least 10% lower. A transfer changes the task assignments and rebuilds both agents' route plans.

This is a small rule-based reallocation experiment, not continuous optimization: the simulation does not model a time-dependent road network or repeatedly search all possible task exchanges. Period metrics describe the resulting plan after each period.

In [ ]:
def simulate_dynamic_rebidding(auctioneer, instance, distance_matrix):
    """Apply cost changes and test the stated one-task transfer rule."""
    print("\n=== DYNAMIC RE-BIDDING SIMULATION ===")
    time_periods = 3
    rebidding_history = []

    for period in range(time_periods):
        print(f"\n--- Time period {period + 1} ---")

        if period == 1:
            print("Congestion scenario: increasing agent distance costs by 30%.")
            for agent in auctioneer.agents:
                agent.cost_per_km *= 1.3
        elif period == 2:
            print("Fuel-price scenario: reducing agent competitiveness by 20%.")
            for agent in auctioneer.agents:
                agent.competitiveness *= 0.8

        rebidding_opportunities = 0
        for agent in auctioneer.agents:
            if len(agent.assigned_tasks) <= 3:
                continue

            for other_agent in auctioneer.agents:
                if (
                    other_agent.agent_id == agent.agent_id
                    or len(other_agent.assigned_tasks) >= 2
                ):
                    continue

                if not agent.assigned_tasks:
                    break

                task_to_trade = agent.assigned_tasks[-1]
                source_tasks = agent.assigned_tasks.copy()
                source_tasks.remove(task_to_trade)
                agent.rebuild_plan(source_tasks)

                source_cost = agent.calculate_bid_cost(task_to_trade)
                receiving_cost = other_agent.calculate_bid_cost(task_to_trade)

                if receiving_cost < source_cost * 0.9:
                    other_tasks = other_agent.assigned_tasks.copy()
                    other_tasks.append(task_to_trade)
                    other_agent.rebuild_plan(other_tasks)
                    task_to_trade.assigned_agent = other_agent.agent_id
                    rebidding_opportunities += 1
                    print(
                        f"Transferred task {task_to_trade.task_id} from "
                        f"Agent {agent.agent_id} to Agent {other_agent.agent_id}."
                    )
                    break

                source_tasks.append(task_to_trade)
                agent.rebuild_plan(source_tasks)

        period_metrics = auctioneer.calculate_system_metrics()
        rebidding_history.append(
            {
                "period": period + 1,
                "metrics": period_metrics,
                "rebidding_opportunities": rebidding_opportunities,
            }
        )
        print(f"Period {period + 1} total distance: "
              f"{period_metrics['total_distance']:.2f} distance units")
        print(f"Period {period + 1} mean current-trip utilization: "
              f"{period_metrics['avg_utilization']:.1f}%")
        print(f"Task transfers in this period: {rebidding_opportunities}")

    return rebidding_history


rebidding_history = simulate_dynamic_rebidding(
    auctioneer,
    instance,
    distance_matrix,
)

## Visualize routes and allocation statistics

The route plot uses node 0 for the depot and nodes 1 through $|C|$ for customers, matching the distance-matrix indexing. Route lines are colored by vehicle and identified in the legend. The panels report route geometry, the number of tasks assigned to each agent, changes in distance and utilization across periods, and a textual summary of system metrics. Distance and utilization use different units, so the two period series have separate vertical axes.

In [ ]:
def visualize_multi_agent_results(
    instance,
    agent_statuses,
    final_metrics,
    rebidding_history,
):
    """Plot routes, task counts, period metrics, and a system summary."""
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(
        2, 2, figsize=(15, 11)
    )

    ax1.scatter(
        instance.depot[0],
        instance.depot[1],
        s=240,
        c="black",
        marker="s",
        label="Depot",
        zorder=5,
    )
    for customer_index, coord in enumerate(instance.customers):
        ax1.scatter(
            coord[0],
            coord[1],
            s=130,
            c="lightgray",
            edgecolors="black",
            zorder=4,
        )
        ax1.annotate(
            f"C{customer_index + 1}\nDemand: {instance.demands[customer_index]}",
            coord,
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
        )

    colors = plt.get_cmap("tab10")
    for status in agent_statuses:
        route = status["route"]
        color = colors(status["agent_id"] % 10)
        for route_index in range(len(route) - 1):
            start_node = route[route_index]
            end_node = route[route_index + 1]
            start_coord = (
                instance.depot
                if start_node == 0
                else instance.customers[start_node - 1]
            )
            end_coord = (
                instance.depot
                if end_node == 0
                else instance.customers[end_node - 1]
            )
            ax1.plot(
                [start_coord[0], end_coord[0]],
                [start_coord[1], end_coord[1]],
                color=color,
                linewidth=2,
                alpha=0.75,
                label=f"Agent {status['agent_id'] + 1}",
            )

    handles, labels = ax1.get_legend_handles_labels()
    unique_legend = dict(zip(labels, handles))
    ax1.legend(unique_legend.values(), unique_legend.keys())
    ax1.set_xlabel("X coordinate (distance units)")
    ax1.set_ylabel("Y coordinate (distance units)")
    ax1.set_title("Planned vehicle routes")
    ax1.grid(True, alpha=0.3)
    ax1.set_aspect("equal", adjustable="datalim")

    agent_labels = [f"Agent {status['agent_id'] + 1}" for status in agent_statuses]
    task_counts = [status["assigned_tasks"] for status in agent_statuses]
    ax2.bar(agent_labels, task_counts, color="steelblue", alpha=0.8)
    ax2.set_xlabel("Vehicle agent")
    ax2.set_ylabel("Assigned tasks (count)")
    ax2.set_title("Task allocation by agent")
    ax2.grid(True, axis="y", alpha=0.3)

    if rebidding_history:
        periods = [entry["period"] for entry in rebidding_history]
        distances = [
            entry["metrics"]["total_distance"] for entry in rebidding_history
        ]
        utilizations = [
            entry["metrics"]["avg_utilization"] for entry in rebidding_history
        ]
        transfers = [
            entry["rebidding_opportunities"] for entry in rebidding_history
        ]

        ax3_twin = ax3.twinx()
        distance_line = ax3.plot(
            periods,
            distances,
            color="tab:blue",
            marker="o",
            label="Total route distance",
        )
        utilization_line = ax3_twin.plot(
            periods,
            utilizations,
            color="tab:red",
            marker="s",
            label="Mean current-trip utilization",
        )
        for period, distance, transfer_count in zip(periods, distances, transfers):
            ax3.annotate(
                f"Transfers: {transfer_count}",
                (period, distance),
                textcoords="offset points",
                xytext=(0, 8),
                ha="center",
                fontsize=8,
            )

        ax3.set_xlabel("Time period")
        ax3.set_ylabel("Total route distance (distance units)", color="tab:blue")
        ax3_twin.set_ylabel("Mean current-trip utilization (%)", color="tab:red")
        ax3.set_title("Metrics under changing agent costs")
        ax3.grid(True, alpha=0.3)
        lines = distance_line + utilization_line
        ax3.legend(lines, [line.get_label() for line in lines], loc="best")
    else:
        ax3.text(0.5, 0.5, "No period results to display", ha="center")
        ax3.set_axis_off()

    ax4.axis("off")
    summary = (
        "System summary\n\n"
        f"Total route distance: {final_metrics['total_distance']:.2f} distance units\n"
        f"Mean current-trip utilization: {final_metrics['avg_utilization']:.1f}%\n"
        f"Task assignment rate: {final_metrics['assignment_rate']:.1f}%\n"
        f"Tasks assigned: {final_metrics['total_tasks_assigned']} / "
        f"{final_metrics['num_tasks']}\n"
        f"Agents: {final_metrics['num_agents']}"
    )
    ax4.text(0.05, 0.9, summary, va="top", fontsize=12, linespacing=1.7)
    ax4.set_title("Reported system metrics")

    fig.tight_layout()
    plt.show()


visualize_multi_agent_results(
    instance,
    agent_statuses,
    final_metrics,
    rebidding_history,
)

## Descriptive analysis of allocation patterns

A multi-agent simulation can produce patterns from repeated local decisions, but a pattern alone is not evidence of optimality or of a general emergent capability. The following statistics describe this run: task-count and utilization variance describe differences between agents, the Gini coefficient summarizes inequality in task counts, and the active-agent fraction reports how many agents received at least one task.

For non-negative task counts $x_1,\ldots,x_n$, the Gini coefficient is

$$
G = \frac{2\sum_{i=1}^{n} i x_{(i)}}{n\sum_{i=1}^{n}x_i} - \frac{n+1}{n},
$$

where $x_{(i)}$ are the counts sorted in non-decreasing order. A value of zero represents equal task counts. If every count is zero, the coefficient is defined as zero. These measures are descriptive indicators for this allocation; they are not a test of route optimality or system resilience under failures.

In [ ]:
def analyze_emergent_behavior(auctioneer, agent_statuses, final_metrics):
    """Calculate descriptive workload and route statistics for this run."""
    print("=== ALLOCATION PATTERN ANALYSIS ===")

    if auctioneer.auction_history:
        wins_per_agent = {}
        for auction in auctioneer.auction_history:
            agent_id = auction["winning_agent"]
            wins_per_agent[agent_id] = wins_per_agent.get(agent_id, 0) + 1

        print(f"\nCompleted task auctions: {len(auctioneer.auction_history)}")
        print("Task wins by agent:")
        for agent_id, wins in sorted(wins_per_agent.items()):
            share = wins / len(auctioneer.auction_history) * 100
            print(f"  Agent {agent_id}: {wins} wins ({share:.1f}%)")

    workloads = [status["assigned_tasks"] for status in agent_statuses]
    utilizations = [status["utilization"] for status in agent_statuses]
    workload_variance = float(np.var(workloads)) if workloads else 0.0
    utilization_variance = float(np.var(utilizations)) if utilizations else 0.0

    print("\nWorkload distribution:")
    print(f"  Task-count variance: {workload_variance:.2f}")
    print(f"  Current-trip utilization variance: {utilization_variance:.2f}")

    total_tasks = final_metrics["total_tasks_assigned"]
    distance_per_task = (
        final_metrics["total_distance"] / total_tasks if total_tasks else 0.0
    )
    print("\nRoute summary:")
    print(f"  Distance per assigned task: {distance_per_task:.2f} distance units")

    for status in agent_statuses:
        if status["assigned_tasks"] == 0:
            label = "No tasks assigned"
        elif status["utilization"] > 80:
            label = "High current-trip utilization"
        elif status["utilization"] > 50:
            label = "Moderate current-trip utilization"
        else:
            label = "Low current-trip utilization"
        print(
            f"  Agent {status['agent_id']}: {label}; "
            f"{status['assigned_tasks']} tasks, "
            f"{status['utilization']:.1f}% current-trip utilization"
        )

    if len(workloads) > 1 and sum(workloads) > 0:
        sorted_workloads = sorted(workloads)
        n_agents = len(sorted_workloads)
        weighted_sum = sum(
            (index + 1) * workload
            for index, workload in enumerate(sorted_workloads)
        )
        gini_coefficient = (
            2 * weighted_sum / (n_agents * sum(sorted_workloads))
            - (n_agents + 1) / n_agents
        )
    else:
        gini_coefficient = 0.0

    active_agents = sum(status["assigned_tasks"] > 0 for status in agent_statuses)
    resilience_indicator = (
        active_agents / len(agent_statuses) if agent_statuses else 0.0
    )

    print("\nDescriptive system indicators:")
    print(f"  Task-count Gini coefficient: {gini_coefficient:.3f}")
    print(
        f"  Active-agent fraction: {resilience_indicator:.1%} "
        f"({active_agents}/{len(agent_statuses)} agents)"
    )

    return {
        "workload_variance": workload_variance,
        "utilization_variance": utilization_variance,
        "distance_per_task": distance_per_task,
        "gini_coefficient": gini_coefficient,
        "resilience": resilience_indicator,
    }


emergent_properties = analyze_emergent_behavior(
    auctioneer,
    agent_statuses,
    final_metrics,
)

## Interpreting the model's capabilities and limits

The code demonstrates decentralized bidding by vehicle agents under a centralized task-allocation authority. Each agent computes a local bid, but the auctioneer collects bids and makes the allocation decision. Therefore, the implementation should not be described as fully decentralized or as having no central point of coordination.

The cost changes and task-transfer rule illustrate adaptation to changed parameters. They do not demonstrate real-time response to observed traffic, fault tolerance, or improved performance relative to a measured baseline. The reported distance is the distance of the route plan produced by this rule; no optimality guarantee is provided. The implementation is auction-based and Contract-Net-inspired, but it does not implement a full Contract Net Protocol with proposal, award, acceptance, and failure-message exchanges.

In [ ]:
def summarize_model_scope(final_metrics, emergent_properties):
    """Report simulation outcomes without claiming an unmeasured comparison."""
    print("=== SIMULATION SUMMARY AND SCOPE ===")
    print(f"Total route distance: {final_metrics['total_distance']:.2f} distance units")
    print(
        "Mean current-trip utilization: "
        f"{final_metrics['avg_utilization']:.1f}%"
    )
    print(f"Task assignment rate: {final_metrics['assignment_rate']:.1f}%")
    print(
        f"Tasks assigned: {final_metrics['total_tasks_assigned']} "
        f"of {final_metrics['num_tasks']}"
    )
    print(
        "Task-count Gini coefficient: "
        f"{emergent_properties['gini_coefficient']:.3f}"
    )
    print(
        "Active-agent fraction: "
        f"{emergent_properties['resilience']:.1%}"
    )
    print("\nInterpretation:")
    print("- Agents submit local cost bids to a centralized auctioneer.")
    print("- Split deliveries are represented by separately allocated tasks.")
    print("- Capacity is enforced per trip, with depot returns between trips.")
    print("- The auction heuristic does not guarantee a globally minimum-distance plan.")
    print("- No external baseline or failure-response experiment is included.")

    return {
        "reported_metrics": final_metrics,
        "descriptive_indicators": emergent_properties,
    }


summary_results = summarize_model_scope(final_metrics, emergent_properties)

## Why this tier is useful

The multi-agent view highlights how local decisions can be coordinated through bids. It makes agent-level costs, eligibility, workload, and route state visible, and offers a simple way to illustrate task transfers when conditions change. These are modeling and teaching benefits; the simulation does not establish that the auction is more efficient than another routing method.

### Advantages represented in this example

- Tasks are allocated through explicit bids rather than by assigning every task to a preselected vehicle.
- Vehicle capacity is tracked across repeated depot-based trips.
- Cost parameters can change between periods, and a simple task-transfer rule can respond to those changes.
- The outcome can be inspected at both agent and system levels.

### Limitations represented in this example

- A central auctioneer remains responsible for collecting bids and awarding tasks.
- The greedy sequential rule does not guarantee a globally optimal routing plan.
- The Euclidean distance model omits roads, travel times, time windows, service times, and operational uncertainty.
- The small fixed instance and simple transfer rule are not evidence of scalability or fault tolerance.
- Communication protocols, strategic bidding, and agent failures are not simulated.

### Appropriate interpretation

Use this notebook to study a small auction-based task-allocation mechanism for an SDVRP instance. Treat its route and workload measures as outputs of the stated heuristic, not as proof of a general performance advantage.

In [ ]:
print("=== KEY RESULTS FROM THIS SIMULATION ===")
print(f"Total route distance: {final_metrics['total_distance']:.2f} distance units")
print(
    "Mean current-trip utilization: "
    f"{final_metrics['avg_utilization']:.1f}%"
)
print(f"Task assignment rate: {final_metrics['assignment_rate']:.1f}%")
print(
    f"Active agents: "
    f"{sum(status['assigned_tasks'] > 0 for status in agent_statuses)}"
    f"/{len(agent_statuses)}"
)
print()
print("Coordination:")
print(f"  Registered agents: {len(auctioneer.agents)}")
print(f"  Tasks created: {final_metrics['num_tasks']}")
print(f"  Task auctions with a winner: {len(auctioneer.auction_history)}")
print(f"  Re-bidding periods: {len(rebidding_history)}")
print(
    "  Task transfers: "
    f"{sum(entry['rebidding_opportunities'] for entry in rebidding_history)}"
)
print()
print("Descriptive allocation indicators:")
print(
    "  Task-count Gini coefficient: "
    f"{emergent_properties['gini_coefficient']:.3f}"
)
print(
    "  Active-agent fraction: "
    f"{emergent_properties['resilience']:.1%}"
)
print(
    "  Distance per assigned task: "
    f"{emergent_properties['distance_per_task']:.2f} distance units"
)
print(
    "  Task-count variance: "
    f"{emergent_properties['workload_variance']:.2f}"
)
print()
print("This is a small, centralized-auction simulation with agent-level bids.")
print("Its outputs describe the heuristic's plan and do not establish global optimality.")